# The Aharonov-Bohm Effect: Classical Wave Analog
## Symplectic Geometry, Gauge Fields, and Topological Phase

This notebook adapts the pseudo-spectral PDE solver to simulate the **Aharonov-Bohm (AB) effect**. While originally a quantum phenomenon where a charged particle is affected by a magnetic potential in a region with zero magnetic field, it has a profound **classical wave analog** (e.g., water waves or electromagnetic waves passing around a localized flux tube).

---

## 1. Physical Setup and Geometry

* **The Flux Tube:** We place a localized magnetic flux tube at the origin $(0,0)$. Outside the origin, the physical magnetic field $\mathbf{B} = \nabla \times \mathbf{A}$ is strictly zero.
* **The Wave Packets:** We initialize two Gaussian wave packets on the left side of the domain. One travels above the origin, and the other travels below it.
* **The Interference:** The packets recombine on the right side. Because they enclose the flux tube, their wavefunctions acquire a relative topological phase shift proportional to the enclosed flux $\Phi$, visibly shifting the interference fringes.

---

## 2. The Governing Equation & Minimal Coupling

In Hamiltonian mechanics and symplectic geometry, a gauge field (vector potential $\mathbf{A}$) couples to the wave via **minimal coupling**. The canonical momentum $\mathbf{p}$ is shifted:
$$
\mathbf{p} \rightarrow \mathbf{p} - q\mathbf{A}
$$
In the Fourier domain (where $\mathbf{p} \sim (\xi, \eta)$), the principal symbol of the spatial operator becomes:
$$
a(x, y, \xi, \eta) = c^2 \left[ (\xi - A_x(x,y))^2 + (\eta - A_y(x,y))^2 \right]
$$

---

## 3. The Vector Potential

For a flux tube of strength $\Phi$ at the origin, the vector potential in the Coulomb gauge is:
$$
A_x(x,y) = -\frac{\Phi}{2\pi} \frac{y}{x^2 + y^2 + \epsilon^2}, \quad A_y(x,y) = \frac{\Phi}{2\pi} \frac{x}{x^2 + y^2 + \epsilon^2}
$$
*(Note: $\epsilon$ is a small regularization parameter to prevent numerical singularity at the origin.)*

---

## 4. Initial Conditions: Split Wave Packets

We initialize two identical Gaussian packets displaced vertically, both given an initial velocity pushing them to the right ($+x$ direction):
$$
u(x,y,0) = \exp\left(-\frac{(x+x_0)^2 + (y-y_0)^2}{2\sigma^2}\right) + \exp\left(-\frac{(x+x_0)^2 + (y+y_0)^2}{2\sigma^2}\right)
$$
$$
\frac{\partial u}{\partial t}(x,y,0) = -c \frac{\partial u}{\partial x}(x,y,0)
$$

# Implementation
## 0. Imports

In [ ]:
from solver import PDESolver, psiOp
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters

In [ ]:
# ── Wave speed ──
C_SQUARED = 1.0        # c² (m²/s²)

# ── Aharonov-Bohm Flux ──
# PHI = 0.0  -> No phase shift (Symmetric interference)
# PHI = 3.14 -> Half flux quantum (Fringes shift by half a period)
PHI = 3.14159          # Enclosed magnetic flux (Webers)
EPS = 0.1              # Regularization parameter to avoid singularity at origin

# ── Grid and Time ──
Lx, Ly   = 10.0, 10.0
# Nx, Ny = 32, 32 
Nx, Ny = 64, 64    
# Nx, Ny = 128, 128    
# Nx, Ny = 128, 256    

Lt, Nt   = 20.0, 400
# Lt, Nt   = 30.0, 600
# Lt, Nt   = 40.0, 800
# Lt, Nt   = 50.0, 1000
# Lt, Nt   = 60.0, 1200
n_frames = 300

## 2. Grid setup

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny)

## 3. SymPy symbols and principal symbol

In [ ]:
x, y, t = sp.symbols('x y t', real=True)
xi, eta = sp.symbols('xi eta', real=True)
u_func  = sp.Function('u')
u       = u_func(t, x, y)

# Vector potential for a flux tube at the origin
r2 = x**2 + y**2 + EPS**2
A_x = -(PHI / (2 * sp.pi)) * y / r2
A_y =  (PHI / (2 * sp.pi)) * x / r2

# Principal symbol with minimal coupling: (p - qA)^2
symbol_ab = C_SQUARED * ((xi - A_x)**2 + (eta - A_y)**2)

print("Principal symbol (Aharonov-Bohm):")
print("  a(x, y, ξ, η) =", symbol_ab)

## 4. Wave equation

In [ ]:
#
#   ∂²u/∂t² = -psiOp(a(ξ), u)
#
equation = sp.Eq(
    sp.diff(u, t, 2),
    -psiOp(symbol_ab, u)
)

print("Equation:")
print(f"  ∂²u/∂t² = -psiOp(a_AB, u)")

## 5. Initial conditions

In [ ]:
def initial_condition_ab(xx, yy):
    """
    Two Gaussian wave packets, one above and one below the origin.
    """
    x0, y0 = -5.0, 2.5  # Center of the upper packet
    sigma = 0.8
    
    # Upper packet
    p1 = np.exp(-((xx - x0)**2 + (yy - y0)**2) / (2 * sigma**2))
    # Lower packet (mirrored across x-axis)
    p2 = np.exp(-((xx - x0)**2 + (yy + y0)**2) / (2 * sigma**2))
    
    return p1 + p2

def initial_velocity_ab(xx, yy):
    """
    Initial velocity pushing both packets to the right (+x direction).
    Based on WKB: v = -c * du/dx
    """
    c = np.sqrt(C_SQUARED)
    x0, y0 = -5.0, 2.5
    sigma = 0.8
    
    # Upper packet derivative
    p1 = np.exp(-((xx - x0)**2 + (yy - y0)**2) / (2 * sigma**2))
    dp1_dx = -(xx - x0) / sigma**2 * p1
    
    # Lower packet derivative
    p2 = np.exp(-((xx - x0)**2 + (yy + y0)**2) / (2 * sigma**2))
    dp2_dx = -(xx - x0) / sigma**2 * p2
    
    return -c * (dp1_dx + dp2_dx)

## 6. Solver setup

In [ ]:
solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet', # Absorb waves at boundaries
    initial_condition=initial_condition_ab,
    initial_velocity=initial_velocity_ab,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve

In [ ]:
frames = solver.solve()

# Plot energy
solver.plot_energy() #(log=True)

## 8. Visualization

In [ ]:
# Raise the animation size limit
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay='contour', # Contours are crucial here to see the phase fringes!
    mode='surface',    
    physical=True      
)

HTML(ani.to_jshtml())

In [ ]:
ani.save('aharonov_bohm_effect.mp4', writer='ffmpeg', fps=20, dpi=100)
print("✅ Saved to aharonov_bohm_effect.mp4")